In [2]:
from pyspark.sql import SparkSession, functions as f

In [3]:
from pathlib import Path
root =  Path.cwd() / 'work'
data_path = root / 'data' / 'raw' / 'infra_ed_basica_2024.csv'
print(data_path)

/home/jovyan/work/data/raw/infra_ed_basica_2024.csv


In [4]:
spark = SparkSession.builder.appName("Analise_INEP").getOrCreate()

In [5]:
df_censo_escolar = spark.read.csv(str(data_path), header=True, sep=';')
df_censo_escolar.show(5)

+------------+---------+---------+--------+-----+-----+--------------------+------------+---------------------+---------------------+-------------------+-------------------+-----------------+--------------+---------------+---------------+--------------------+-----------+--------------------+-----------+--------------+---------------------------+--------------+---------------------------+--------------------+-----------+---------------+-----------+--------+------+-----------+-------------------------+-----------------+--------------------+---------------------+------------------------------+----------------------------+---------------------------+----------------------+-------------------------+-------------------------+----------------------------+---------------------------+-------------------------+----------------------------+--------------------------+---------------------------+----------------------------+------------------------------+----------------------------+---------------

In [6]:
df_censo_escolar = df_censo_escolar.filter((f.col('TP_SITUACAO_FUNCIONAMENTO') == '1'))

In [7]:
prefixos_infra = ('IN_AGUA', 'IN_ENERGIA', 'IN_ESGOTO', 'IN_LIXO', 'IN_INTERNET', 'IN_BANHEIRO', 'IN_ACESSABILIDADE', 'IN_COZINHA', 'IN_REFEITORIO', 'IN_BIBLIOTECA', 'IN_QUADRA_ESPORTES', 'IN_ALIMENTACAO', 'IN_LABORATORIO')
colunas = df_censo_escolar.columns

colunas_infra = []
for coluna in colunas:
    if coluna.startswith(prefixos_infra):
        colunas_infra.append(coluna)

colunas_infra = ['NO_REGIAO','CO_MUNICIPIO', 'NO_MUNICIPIO', 'SG_UF'] + colunas_infra
df_censo_escolar = df_censo_escolar.select(colunas_infra)
df_censo_escolar.show(5)

+---------+------------+--------------------+-----+---------------+--------------------+----------------------+---------------+-----------------+-------------------+------------------+-----------------------+-------------------------+--------------------+----------------------+----------------------+-----------------------+---------------------+---------------+---------------------+----------------------+--------------+---------------+-----------------------------+---------------------------+-----------+--------------+---------------+------------------------+--------------------+-------------+--------------------------+----------+-----------------------+--------------------------+------------------------+------------------+--------------------------+-----------------------------+-------------+-----------+------------------+--------------------------+------------------------+----------------------+--------------+
|NO_REGIAO|CO_MUNICIPIO|        NO_MUNICIPIO|SG_UF|IN_AGUA_POTAVEL|IN_AGUA

In [8]:
df_censo_escolar.dropDuplicates(['NO_REGIAO']).show()

+------------+------------+--------------------+-----+---------------+--------------------+----------------------+---------------+-----------------+-------------------+------------------+-----------------------+-------------------------+--------------------+----------------------+----------------------+-----------------------+---------------------+---------------+---------------------+----------------------+--------------+---------------+-----------------------------+---------------------------+-----------+--------------+---------------+------------------------+--------------------+-------------+--------------------------+----------+-----------------------+--------------------------+------------------------+------------------+--------------------------+-----------------------------+-------------+-----------+------------------+--------------------------+------------------------+----------------------+--------------+
|   NO_REGIAO|CO_MUNICIPIO|        NO_MUNICIPIO|SG_UF|IN_AGUA_POTAVEL|I

In [ ]:
df_censo_escolar = df_censo_escolar.filter((f.col('NO_REGIAO').isin('Nordeste', 'Sudeste')))
df_censo_escolar.show(5)

+---------+------------+------------+-----+---------------+--------------------+----------------------+---------------+-----------------+-------------------+------------------+-----------------------+-------------------------+--------------------+----------------------+----------------------+-----------------------+---------------------+---------------+---------------------+----------------------+--------------+---------------+-----------------------------+---------------------------+-----------+--------------+---------------+------------------------+--------------------+-------------+--------------------------+----------+-----------------------+--------------------------+------------------------+------------------+--------------------------+-----------------------------+-------------+-----------+------------------+--------------------------+------------------------+----------------------+--------------+
|NO_REGIAO|CO_MUNICIPIO|NO_MUNICIPIO|SG_UF|IN_AGUA_POTAVEL|IN_AGUA_REDE_PUBLICA|IN

In [ ]:
df_censo_escolar.printSchema()

root
 |-- NO_REGIAO: string (nullable = true)
 |-- CO_MUNICIPIO: string (nullable = true)
 |-- NO_MUNICIPIO: string (nullable = true)
 |-- SG_UF: string (nullable = true)
 |-- IN_AGUA_POTAVEL: string (nullable = true)
 |-- IN_AGUA_REDE_PUBLICA: string (nullable = true)
 |-- IN_AGUA_POCO_ARTESIANO: string (nullable = true)
 |-- IN_AGUA_CACIMBA: string (nullable = true)
 |-- IN_AGUA_FONTE_RIO: string (nullable = true)
 |-- IN_AGUA_INEXISTENTE: string (nullable = true)
 |-- IN_AGUA_CARRO_PIPA: string (nullable = true)
 |-- IN_ENERGIA_REDE_PUBLICA: string (nullable = true)
 |-- IN_ENERGIA_GERADOR_FOSSIL: string (nullable = true)
 |-- IN_ENERGIA_RENOVAVEL: string (nullable = true)
 |-- IN_ENERGIA_INEXISTENTE: string (nullable = true)
 |-- IN_ESGOTO_REDE_PUBLICA: string (nullable = true)
 |-- IN_ESGOTO_FOSSA_SEPTICA: string (nullable = true)
 |-- IN_ESGOTO_FOSSA_COMUM: string (nullable = true)
 |-- IN_ESGOTO_FOSSA: string (nullable = true)
 |-- IN_ESGOTO_INEXISTENTE: string (nullable = true)

In [ ]:
colunas = [f.col(c).cast('integer').alias(c) if c.startswith('IN_') else f.col(c)for c in df_censo_escolar.columns]
df_censo_escolar = df_censo_escolar.select(colunas)

In [ ]:
df_censo_escolar.printSchema()

root
 |-- NO_REGIAO: string (nullable = true)
 |-- CO_MUNICIPIO: string (nullable = true)
 |-- NO_MUNICIPIO: string (nullable = true)
 |-- SG_UF: string (nullable = true)
 |-- IN_AGUA_POTAVEL: integer (nullable = true)
 |-- IN_AGUA_REDE_PUBLICA: integer (nullable = true)
 |-- IN_AGUA_POCO_ARTESIANO: integer (nullable = true)
 |-- IN_AGUA_CACIMBA: integer (nullable = true)
 |-- IN_AGUA_FONTE_RIO: integer (nullable = true)
 |-- IN_AGUA_INEXISTENTE: integer (nullable = true)
 |-- IN_AGUA_CARRO_PIPA: integer (nullable = true)
 |-- IN_ENERGIA_REDE_PUBLICA: integer (nullable = true)
 |-- IN_ENERGIA_GERADOR_FOSSIL: integer (nullable = true)
 |-- IN_ENERGIA_RENOVAVEL: integer (nullable = true)
 |-- IN_ENERGIA_INEXISTENTE: integer (nullable = true)
 |-- IN_ESGOTO_REDE_PUBLICA: integer (nullable = true)
 |-- IN_ESGOTO_FOSSA_SEPTICA: integer (nullable = true)
 |-- IN_ESGOTO_FOSSA_COMUM: integer (nullable = true)
 |-- IN_ESGOTO_FOSSA: integer (nullable = true)
 |-- IN_ESGOTO_INEXISTENTE: integer (

In [ ]:
df_censo_escolar.groupBy(df_censo_escolar['CO_MUNICIPIO'])\
    .count()\
    .where(f.col('count') > 1)\
    .select(f.sum('count'))\
    .show()

+----------+
|sum(count)|
+----------+
|    121873|
+----------+



In [ ]:
df_censo_escolar.select([f.count(f.when(f.isnan(c) | f.col(c).isNull(), c)).alias(c) for c in df_censo_escolar.columns]).show()

+---------+------------+------------+-----+---------------+--------------------+----------------------+---------------+-----------------+-------------------+------------------+-----------------------+-------------------------+--------------------+----------------------+----------------------+-----------------------+---------------------+---------------+---------------------+----------------------+--------------+---------------+-----------------------------+---------------------------+-----------+--------------+---------------+------------------------+--------------------+-------------+--------------------------+----------+-----------------------+--------------------------+------------------------+------------------+--------------------------+-----------------------------+-------------+-----------+------------------+--------------------------+------------------------+----------------------+--------------+
|NO_REGIAO|CO_MUNICIPIO|NO_MUNICIPIO|SG_UF|IN_AGUA_POTAVEL|IN_AGUA_REDE_PUBLICA|IN